### Aula 02 — Lineage, DAG e Catalyst

Nesta aula, o foco é acompanhar como uma sequência de transformações se torna um plano otimizado. Serão comparados filtros separados e combinados, além de uma projeção com coluna calculada. As evidências são os planos já salvos: não há `show()` ou `count()` para materializar os resultados das consultas construídas aqui.

#### 1. Sessão e expressões de coluna

As próximas células importam `SparkSession` e as funções SQL como `F`, e obtêm a sessão `aula02_lineage_dag`. `F.col()` produz expressões que o Spark consegue analisar; as comparações descrevem condições sobre registros distribuídos, sem percorrê-los em um laço Python.


In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [2]:
spark = (
    SparkSession.builder
    .appName("aula02_lineage_dag")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/11 13:04:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


#### 2. Fonte de dados e avaliação adiada

A leitura aponta para `02_lineage_DAG/clientes.csv`, com cabeçalho e inferência de tipos. O arquivo origina a relação usada por todos os exemplos seguintes. Os planos salvos identificam seis campos: `cliente_id`, `nome`, `idade`, `cidade`, `renda_mensal` e `ativo`.

As transformações subsequentes são lazy: constroem descrições de processamento para avaliação quando uma action solicitar dados. A inferência de schema da leitura pode acessar a fonte antes disso, pois precisa descobrir os tipos; ela não materializa automaticamente os resultados de `df2`, `df_a` ou `df_b`.


In [3]:
df = spark.read.csv("/workspace/datasets/02_lineage_DAG/clientes.csv", header= True, inferSchema= True)

#### 3. Lineage: dependências entre transformações

`df2` parte de `df`, mantém clientes com `idade > 30`, seleciona três campos e cria `idade_futura` somando um à idade. O cálculo depende de `idade` continuar disponível depois do `select()`. Cada transformação devolve outro DataFrame; a fonte e `df` permanecem como referências para outros ramos do estudo.

**Lineage** é o encadeamento de dependências que permite reconstruir um resultado a partir da origem. **DAG** é um grafo direcionado sem ciclos: as dependências orientam a ordem em que o trabalho pode ocorrer. A árvore de planos SQL impressa por `explain` e o DAG de jobs/stages da execução representam níveis distintos. Portanto, três chamadas de transformação não significam necessariamente três stages ou três leituras do CSV.

Em Engenharia de Dados, explicitar essas dependências ajuda a rastrear de onde veio cada campo e a entender o que precisaria ser recomputado. O DAG efetivamente executado não é mostrado neste notebook; isso exigiria evidências de uma action e do acompanhamento da execução.


In [4]:
df2 = (
    df.filter(F.col("idade") > 30)
    .select("nome", "idade", "cidade")
    .withColumn("idade_futura", F.col("idade") + 1)
)

#### 4. Experimento: três filtros separados ou uma condição composta

As próximas duas células mantêm a mesma regra: idade acima de 30, cliente ativo e renda mensal acima de 5.000. `df_a` expressa as condições em três chamadas; `df_b` usa uma chamada com `&`, que representa AND entre expressões `Column`. Os parênteses delimitam cada comparação.

Como essas condições são determinísticas e usam as mesmas colunas, espera-se que o otimizador consiga combiná-las sem mudar os registros elegíveis. O experimento verifica essa expectativa pelos planos, e não por tempo medido. Isso evita concluir que uma cadeia com mais chamadas Python necessariamente processa mais vezes os mesmos dados.


In [5]:
df_a = (
    df
    .filter(F.col("idade") > 30)
    .filter(F.col("ativo") == True)
    .filter(F.col("renda_mensal") > 5000)
)

In [6]:
df_b = (
    df.filter(
        (F.col("idade") > 30) &
        (F.col("ativo") == True) &
        (F.col("renda_mensal") > 5000)
    )
)

#### 5. Como ler os quatro planos de `explain(True)`

A chamada estendida apresenta etapas de planejamento:

| Etapa | O que observar neste exemplo |
| --- | --- |
| `Parsed Logical Plan` | Três nós `Filter` refletem a construção de `df_a`. Referências ainda não resolvidas podem ter apóstrofos. |
| `Analyzed Logical Plan` | Colunas e tipos estão resolvidos; `5000` é convertido para `double`, compatível com `renda_mensal`. |
| `Optimized Logical Plan` | As condições passam a um único filtro, com testes `isnotnull` e simplificação de `ativo == True`. |
| `Physical Plan` | O Spark escolhe `FileScan csv` seguido de `Filter` para executar a consulta. |

O Catalyst participa da análise e das otimizações sobre os planos estruturados. A passagem ao plano físico escolhe os operadores executáveis; `explain(True)` permite inspecionar esse processo, sem medir seu custo real. Leia a árvore física de baixo para cima, seguindo a entrada dos dados.

No plano salvo, `PushedFilters` registra predicados enviados ao leitor CSV, enquanto o operador `Filter` continua presente. Isso não comprova eliminação de arquivos ou blocos físicos. `PartitionFilters: []` informa que não há filtro de partições de armazenamento nesta leitura. A marca `*(1)` identifica um trecho com geração conjunta de código; não é a quantidade de partições ou de tasks.


In [7]:
df_a.explain(True)

== Parsed Logical Plan ==
'Filter ('renda_mensal > 5000)
+- Filter (ativo#22 = true)
   +- Filter (idade#19 > 30)
      +- Relation [cliente_id#17,nome#18,idade#19,cidade#20,renda_mensal#21,ativo#22] csv

== Analyzed Logical Plan ==
cliente_id: int, nome: string, idade: int, cidade: string, renda_mensal: double, ativo: boolean
Filter (renda_mensal#21 > cast(5000 as double))
+- Filter (ativo#22 = true)
   +- Filter (idade#19 > 30)
      +- Relation [cliente_id#17,nome#18,idade#19,cidade#20,renda_mensal#21,ativo#22] csv

== Optimized Logical Plan ==
Filter (((isnotnull(idade#19) AND isnotnull(ativo#22)) AND isnotnull(renda_mensal#21)) AND ((idade#19 > 30) AND (ativo#22 AND (renda_mensal#21 > 5000.0))))
+- Relation [cliente_id#17,nome#18,idade#19,cidade#20,renda_mensal#21,ativo#22] csv

== Physical Plan ==
*(1) Filter (((((isnotnull(idade#19) AND isnotnull(ativo#22)) AND isnotnull(renda_mensal#21)) AND (idade#19 > 30)) AND ativo#22) AND (renda_mensal#21 > 5000.0))
+- FileScan csv [cliente

##### 5.1. Comparação observada de `df_a` e `df_b`

A saída seguinte começa com um único `Filter` no plano sintático, diferente dos três filtros de `df_a`. Após a otimização, as duas consultas combinam as mesmas condições e apresentam o **mesmo texto de operadores e predicados no plano físico salvo**: `FileScan csv` e um `Filter` composto.

Os testes `isnotnull` são compatíveis com as comparações usadas: uma linha só permanece quando a condição do filtro é verdadeira. Em particular, a comparação com valor nulo não é automaticamente verdadeira. Essa semelhança física sustenta a equivalência de estratégia neste experimento; não é uma prova de igualdade de tempo, nem autoriza trocar arbitrariamente a ordem de qualquer transformação.


In [8]:
df_b.explain(True)

== Parsed Logical Plan ==
'Filter ((('idade > 30) AND ('ativo = true)) AND ('renda_mensal > 5000))
+- Relation [cliente_id#17,nome#18,idade#19,cidade#20,renda_mensal#21,ativo#22] csv

== Analyzed Logical Plan ==
cliente_id: int, nome: string, idade: int, cidade: string, renda_mensal: double, ativo: boolean
Filter (((idade#19 > 30) AND (ativo#22 = true)) AND (renda_mensal#21 > cast(5000 as double)))
+- Relation [cliente_id#17,nome#18,idade#19,cidade#20,renda_mensal#21,ativo#22] csv

== Optimized Logical Plan ==
Filter (((isnotnull(idade#19) AND isnotnull(ativo#22)) AND isnotnull(renda_mensal#21)) AND (((idade#19 > 30) AND ativo#22) AND (renda_mensal#21 > 5000.0)))
+- Relation [cliente_id#17,nome#18,idade#19,cidade#20,renda_mensal#21,ativo#22] csv

== Physical Plan ==
*(1) Filter (((((isnotnull(idade#19) AND isnotnull(ativo#22)) AND isnotnull(renda_mensal#21)) AND (idade#19 > 30)) AND ativo#22) AND (renda_mensal#21 > 5000.0))
+- FileScan csv [cliente_id#17,nome#18,idade#19,cidade#20,rend

#### 6. Representação de `df2` sem apresentar registros

`print(df2)` mostra os campos resultantes: `nome`, `idade`, `cidade` e `idade_futura`, com os respectivos tipos. A saída salva confirma o schema descrito pela cadeia de transformações. Ela não mostra clientes específicos nem avalia se todas as linhas obedecem às expectativas de qualidade.


In [6]:
print(df2)

DataFrame[nome: string, idade: int, cidade: string, idade_futura: int]


##### 6.1. Projeção, cálculo e redução de colunas no plano

O plano inicial de `df2` contém dois `Project`: um seleciona os campos e o seguinte acrescenta `idade_futura`. No plano otimizado salvo, eles são combinados em uma projeção com `(idade + 1) AS idade_futura`, acima do filtro de idade.

No plano físico, o fluxo observado é `FileScan csv → Filter → Project`. O `ReadSchema` solicita apenas `nome`, `idade` e `cidade`; campos que não participam da consulta foram removidos da leitura lógica. Essa é uma evidência de **Column Pruning**. Como a fonte é CSV, reduzir os campos interpretados e materializados não equivale à leitura de blocos colunares independentes de um formato como Parquet.

Não aparece `Exchange`: as operações mostradas podem avaliar e projetar cada registro na própria partição, sem demandar redistribuição entre partições. Os planos demonstram a estratégia escolhida; não há contagem de linhas, comparação de duração ou DAG executado salvo para essa consulta.


In [7]:
df2.explain(True)

== Parsed Logical Plan ==
'Project [nome#18, idade#19, cidade#20, ('idade + 1) AS idade_futura#33]
+- Project [nome#18, idade#19, cidade#20]
   +- Filter (idade#19 > 30)
      +- Relation [cliente_id#17,nome#18,idade#19,cidade#20,renda_mensal#21,ativo#22] csv

== Analyzed Logical Plan ==
nome: string, idade: int, cidade: string, idade_futura: int
Project [nome#18, idade#19, cidade#20, (idade#19 + 1) AS idade_futura#33]
+- Project [nome#18, idade#19, cidade#20]
   +- Filter (idade#19 > 30)
      +- Relation [cliente_id#17,nome#18,idade#19,cidade#20,renda_mensal#21,ativo#22] csv

== Optimized Logical Plan ==
Project [nome#18, idade#19, cidade#20, (idade#19 + 1) AS idade_futura#33]
+- Filter (isnotnull(idade#19) AND (idade#19 > 30))
   +- Relation [cliente_id#17,nome#18,idade#19,cidade#20,renda_mensal#21,ativo#22] csv

== Physical Plan ==
*(1) Project [nome#18, idade#19, cidade#20, (idade#19 + 1) AS idade_futura#33]
+- *(1) Filter (isnotnull(idade#19) AND (idade#19 > 30))
   +- FileScan c

#### 7. Encerramento

`spark.stop()` encerra a sessão após a inspeção. O trabalho documentado neste notebook compara planos, sem acrescentar uma action de materialização dos DataFrames transformados.


In [8]:
spark.stop()

#### Resumo dos aprendizados

- Lineage descreve dependências; planos SQL e DAG de execução ajudam a investigá-las em níveis diferentes.
- Transformações constroem uma consulta que pode ser otimizada antes de uma action solicitar resultados.
- Filtros separados e combinados produziram a mesma estratégia física neste exemplo.
- A análise resolve nomes e tipos; a otimização combina filtros, simplifica expressões e reduz projeções.
- `df2` mostrou uma projeção calculada e leitura lógica de apenas três campos do CSV.
- Planos semelhantes não são uma medição de desempenho nem um registro completo de execução.

Para aprofundar: [documentação da Aula 02](../docs/02_lineage_dag.md) e [referência de `explain`](https://spark.apache.org/docs/3.5.8/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.explain.html).
